<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 16px; text-align: center; margin-bottom: 20px;">
  <h1 style="color: #e94560; font-size: 2.4em; margin: 0; font-family: 'Segoe UI', sans-serif;">🎙️ Moshi + Bridge + Ditto</h1>
  <h2 style="color: #a4b9d3; font-size: 1.3em; font-weight: 300; margin: 10px 0 0 0;">Unified Talking-Head Video Pipeline</h2>
  <p style="color: #6b8cae; margin: 15px 0 0 0; font-size: 0.95em;">
    Audio Input → Moshi LM → Bridge Module → Ditto Diffusion → 🎬 Talking-Head Video with Moshi Voice
  </p>
</div>

---

### 📋 Pipeline Overview

| Step | Component | Input → Output | Rate |
|------|-----------|----------------|------|
| 1 | **Moshi LM** | Audio → acoustic tokens + response WAV | 12.5 Hz tokens |
| 2 | **Bridge Module** | Tokens (T×8) → HuBERT features (N×1024) | 25 Hz features |
| 3 | **Ditto Diffusion** | Features + Portrait → silent .mp4 | 25 FPS |
| 4 | **FFmpeg** | Silent video + Moshi WAV → final .mp4 | — |

> **Requirements**: CUDA GPU (Ampere or newer for TRT), all checkpoints pre-downloaded.

---
## 🔧 Cell 1 — Install All Dependencies

Run this **once** at the start of your RunPod session. Takes ~3–5 minutes.

In [ ]:
!git clone https://github.com/B-I-T-W-I-S-E-M-I-N-D-S/moshi-ditto-head-pause.git

In [ ]:
%cd moshi-ditto-head-pause

In [ ]:
import os, subprocess, sys

# Find project root (directory containing this notebook)
PROJECT_ROOT = os.path.abspath(os.getcwd())
print(f"📁 Project root: {PROJECT_ROOT}")

# Run the install script
install_script = os.path.join(PROJECT_ROOT, "install.sh")
if not os.path.isfile(install_script):
    raise FileNotFoundError(f"install.sh not found at: {install_script}")

print("🚀 Starting installation...\n")
result = subprocess.run(
    ["bash", install_script],
    cwd=PROJECT_ROOT,
    text=True,
)
if result.returncode != 0:
    print("❌ Installation failed — check output above.")
else:
    print("\n✅ Installation complete! You may now run the pipeline cells below.")

In [ ]:
!apt update
!apt-get install -y ffmpeg libcudnn8 libcudnn8-dev

In [ ]:
!pip install filetype cython huggingface_hub
!pip install cuda-python==12.4.0
!python -c "from cuda import cuda, cudart, nvrtc; print('✅ cuda-python 12.4 working')"
!pip install tensorrt==8.6.1 --extra-index-url https://pypi.nvidia.com
!python -c "import tensorrt as trt; print('TRT:', trt.__version__)"

---
## ✅ Cell 2 — Verify GPU & Environment

In [ ]:
import torch
import shutil
import subprocess

print("=" * 55)
print("  Environment Check")
print("=" * 55)

# Python
print(f"  Python       : {sys.version.split()[0]}")

# PyTorch + CUDA
print(f"  PyTorch      : {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU          : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM         : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"  CUDA version : {torch.version.cuda}")

# FFmpeg
ffmpeg_path = shutil.which("ffmpeg")
if ffmpeg_path:
    ffv = subprocess.check_output(["ffmpeg", "-version"], text=True).split('\n')[0]
    print(f"  FFmpeg       : {ffv}")
else:
    print("  FFmpeg       : ❌ NOT FOUND — install with: apt-get install -y ffmpeg")

print("=" * 55)

---
## ⚙️ Cell 3 — Configuration

**Edit the paths below to match your RunPod setup.**

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "Darknsu/librispeech-full-dataset-model"
download_dir = "./checkpoints"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading bridge_best.pt...")

snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    local_dir=download_dir,
    local_dir_use_symlinks=False,
    allow_patterns=["bridge_best.pt"]  # ✅ only this file
)

print("✅ File downloaded to:", download_dir)

In [ ]:
from huggingface_hub import snapshot_download
import os

repo_id = "digital-avatar/ditto-talkinghead"
download_dir = "./ditto-inference/checkpoints"

# Create directory if it doesn't exist
os.makedirs(download_dir, exist_ok=True)

print("📥 Downloading full repository...")

snapshot_download(
    repo_id=repo_id,
    repo_type="model",   # important
    local_dir=download_dir,
    local_dir_use_symlinks=False,  # ensures real files, not symlinks
)

print("✅ Repo downloaded to:", download_dir)

In [ ]:
!ls ./checkpoints/ditto_cfg

In [ ]:
import os, sys
import torch

# ── Project root (auto-detected) ─────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.getcwd())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"📁 Project root: {PROJECT_ROOT}")

# ════════════════════════════════════════════════════
# ✏️  USER INPUTS — Edit these two paths
# ════════════════════════════════════════════════════
AUDIO_INPUT  = "1.wav"       # Your input audio file
IMAGE_INPUT  = "obama.jpeg"    # Your portrait image
OUTPUT_PATH  = "output_final.mp4" # Where to save the result

# ════════════════════════════════════════════════════
# 🔧 Model Paths — Adjust if your layout differs
# ════════════════════════════════════════════════════

# Bridge
BRIDGE_CKPT   = os.path.join(PROJECT_ROOT, "checkpoints", "bridge_best.pt")
BRIDGE_CONFIG = os.path.join(PROJECT_ROOT, "bridge_module", "config.yaml")

# Ditto TRT models (pre-loaded on RunPod)
DITTO_DATA_ROOT = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_trt_Ampere_Plus")
DITTO_CFG_PKL   = os.path.join(PROJECT_ROOT, "ditto-inference",
                                "checkpoints", "ditto_cfg",
                                "v0.4_hubert_cfg_trt.pkl")

# Moshi model (auto-downloaded from HuggingFace if not cached)
MOSHI_HF_REPO  = "kyutai/moshiko-pytorch-bf16"
MOSHI_WEIGHT   = None   # Set to local path to skip HF download, e.g. "/workspace/model.safetensors"
MIMI_WEIGHT    = None   # Set to local path to skip HF download
MOSHI_TOKENIZER = None  # Set to local path to skip HF download

# ════════════════════════════════════════════════════
# ⚙️  Runtime Settings
# ════════════════════════════════════════════════════
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE        = torch.bfloat16    # Use torch.float16 for older GPUs
BATCH_SIZE   = 1                  # Moshi default; keep as-is for best quality
BATCH_INDEX  = 0                  # Which of the 8 generated responses to use

# ════════════════════════════════════════════════════
# 📂  Optional: save intermediate files
# ════════════════════════════════════════════════════
SAVE_MOSHI_AUDIO       = None    # e.g. "/workspace/moshi_audio.wav"
SAVE_BRIDGE_FEATURES   = None    # e.g. "/workspace/bridge_features.npy"
SAVE_SILENT_VIDEO      = None    # e.g. "/workspace/silent_video.mp4"

# ── Validate paths ────────────────────────────────────────────────────────────
print("\n📋 Configuration Summary")
print("─" * 55)

errors = []
for label, path in [
    ("Audio input",    AUDIO_INPUT),
    ("Image input",    IMAGE_INPUT),
    ("Bridge ckpt",    BRIDGE_CKPT),
    ("Bridge config",  BRIDGE_CONFIG),
    ("Ditto data root", DITTO_DATA_ROOT),
    ("Ditto cfg pkl",  DITTO_CFG_PKL),
]:
    exists = os.path.exists(path)
    status = "✅" if exists else "❌"
    print(f"  {status}  {label:<20} {path}")
    if not exists:
        errors.append(f"{label}: {path}")

print(f"\n  Device : {DEVICE}")
print(f"  Dtype  : {DTYPE}")
print(f"  Batch  : {BATCH_SIZE} (using index {BATCH_INDEX})")

if errors:
    print(f"\n⚠️  {len(errors)} path(s) not found — fix before running the pipeline:")
    for e in errors:
        print(f"   • {e}")
else:
    print("\n✅ All paths verified. Ready to run!")

---
## 🚀 Cell 4 — Load Models

Loads all models into GPU memory. **Run once per session** — no need to reload between different inputs.

In [ ]:
# import gc, torch
# gc.collect()
# torch.cuda.empty_cache()
# torch.cuda.reset_peak_memory_stats()
# print(f"Free VRAM: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

In [ ]:
import time
from pipeline.moshi_runner import MoshiTokenRunner
from pipeline.bridge_runner import BridgeRunner
from pipeline.ditto_runner import DittoRunner
from pipeline.merge_audio_video import merge_audio_into_video

print("═" * 55)
print("  Loading Models into GPU Memory")
print("═" * 55)

# ── Moshi (downloads from HF on first run, then cached) ──────────────────────
print("\n[1/3] 🎙️  Loading Moshi + Mimi...")
t0 = time.time()
moshi_runner = MoshiTokenRunner(
    hf_repo      = MOSHI_HF_REPO,
    moshi_weight = MOSHI_WEIGHT,
    mimi_weight  = MIMI_WEIGHT,
    tokenizer    = MOSHI_TOKENIZER,
    device       = DEVICE,
    dtype        = DTYPE,
    batch_size   = BATCH_SIZE,
)
print(f"    ✅ Moshi ready ({time.time()-t0:.1f}s)")

# ── Bridge Module ─────────────────────────────────────────────────────────────
print("\n[2/3] 🔗  Loading Bridge module...")
t0 = time.time()
bridge_runner = BridgeRunner(
    checkpoint_path = BRIDGE_CKPT,
    config_path     = BRIDGE_CONFIG,
    device          = DEVICE,
)
print(f"    ✅ Bridge ready ({time.time()-t0:.1f}s)")

# ── Ditto SDK (loads all TRT engines) ─────────────────────────────────────────
print("\n[3/3] 🎬  Loading Ditto TRT SDK...")
t0 = time.time()
ditto_runner = DittoRunner(
    data_root = DITTO_DATA_ROOT,
    cfg_pkl   = DITTO_CFG_PKL,
)
print(f"    ✅ Ditto ready ({time.time()-t0:.1f}s)")

print("\n" + "═" * 55)
print("  ✅ All models loaded! Proceed to Cell 5 to run inference.")
print("═" * 55)

---
## 🎬 Cell 5 — Run the Full Pipeline

This cell executes all 4 pipeline steps end-to-end automatically.

In [ ]:
import os, time, tempfile
import torch
import numpy as np

pipeline_start = time.time()

print("═" * 60)
print("  Moshi + Bridge + Ditto — Starting Pipeline")
print("═" * 60)
print(f"  🎙️  Audio : {AUDIO_INPUT}")
print(f"  🖼️  Image : {IMAGE_INPUT}")
print(f"  📹  Output: {OUTPUT_PATH}")
print("═" * 60)

# ══════════════════════════════════════════════════════════
# STEP 1 — Moshi Inference
# User audio → Moshi LM → acoustic tokens (T,8) + WAV
# ══════════════════════════════════════════════════════════
print("\n[Step 1/4] 🎙️  Moshi inference...")
t0 = time.time()

# Determine where to save Moshi audio
if SAVE_MOSHI_AUDIO:
    moshi_audio_dest = SAVE_MOSHI_AUDIO
else:
    _tmp_audio = tempfile.NamedTemporaryFile(suffix="_moshi.wav", delete=False)
    moshi_audio_dest = _tmp_audio.name
    _tmp_audio.close()

with torch.no_grad():
    moshi_audio_path, acoustic_tokens = moshi_runner.run(
        audio_input_path  = AUDIO_INPUT,
        batch_index       = BATCH_INDEX,
        output_audio_path = moshi_audio_dest,
    )

step1_time = time.time() - t0
print(f"  ✅ Done ({step1_time:.1f}s)")
print(f"     Tokens : {tuple(acoustic_tokens.shape)}  dtype={acoustic_tokens.dtype}")
print(f"     Audio  : {moshi_audio_path}")

# ══════════════════════════════════════════════════════════
# STEP 2 — Bridge Module
# Acoustic tokens (T, 8) → HuBERT-like features (N, 1024)
# ══════════════════════════════════════════════════════════
print("\n[Step 2/4] 🔗  Bridge module (tokens → features)...")
t0 = time.time()

audio_features = bridge_runner.run(acoustic_tokens)   # (N, 1024) float32 numpy

if SAVE_BRIDGE_FEATURES:
    np.save(SAVE_BRIDGE_FEATURES, audio_features)
    print(f"     Saved bridge features → {SAVE_BRIDGE_FEATURES}")

step2_time = time.time() - t0
print(f"  ✅ Done ({step2_time:.1f}s)")
print(f"     Features: {audio_features.shape}  dtype={audio_features.dtype}")
print(f"     Duration: {len(audio_features)/25.0:.1f}s @ 25 Hz")

# ══════════════════════════════════════════════════════════
# STEP 3 — Ditto Video Generation
# Portrait image + features (N, 1024) → silent .mp4
# ══════════════════════════════════════════════════════════
print("\n[Step 3/4] 🎬  Ditto talking-head generation...")
t0 = time.time()

if SAVE_SILENT_VIDEO:
    silent_video_path = SAVE_SILENT_VIDEO
else:
    _tmp_silent = tempfile.NamedTemporaryFile(suffix="_silent.mp4", delete=False)
    silent_video_path = _tmp_silent.name
    _tmp_silent.close()
    os.unlink(silent_video_path)   # remove empty file; Ditto will write it

silent_video_path = ditto_runner.run(
    image_path     = IMAGE_INPUT,
    audio_features = audio_features,
    output_path    = silent_video_path,
)

step3_time = time.time() - t0
print(f"  ✅ Done ({step3_time:.1f}s)")
print(f"     Silent video: {silent_video_path}")

# ══════════════════════════════════════════════════════════
# STEP 4 — Merge Moshi Audio + Ditto Video
# silent .mp4 + moshi .wav → final .mp4 with AI voice
# ══════════════════════════════════════════════════════════
print("\n[Step 4/4] 🔊  Merging Moshi audio into video...")
t0 = time.time()

final_path = merge_audio_into_video(
    video_path  = silent_video_path,
    audio_path  = moshi_audio_path,
    output_path = OUTPUT_PATH,
    overwrite   = True,
)

step4_time = time.time() - t0
print(f"  ✅ Done ({step4_time:.1f}s)")

# ── Cleanup temp files ────────────────────────────────────────────────────────
if not SAVE_MOSHI_AUDIO and os.path.isfile(moshi_audio_dest):
    os.unlink(moshi_audio_dest)
if not SAVE_SILENT_VIDEO and os.path.isfile(silent_video_path):
    os.unlink(silent_video_path)

# ── Summary ───────────────────────────────────────────────────────────────────
total_time = time.time() - pipeline_start
print("\n" + "═" * 60)
print(f"  ✅  Pipeline complete in {total_time:.1f}s")
print(f"      Step 1 (Moshi)   : {step1_time:.1f}s")
print(f"      Step 2 (Bridge)  : {step2_time:.1f}s")
print(f"      Step 3 (Ditto)   : {step3_time:.1f}s")
print(f"      Step 4 (FFmpeg)  : {step4_time:.1f}s")
print(f"  📹  Output → {final_path}")
print("═" * 60)

---
## 🎥 Cell 6 — Preview Output Video (Inline)

In [ ]:
import os
from IPython.display import Video, HTML, display

if not os.path.isfile(OUTPUT_PATH):
    print(f"❌ Output file not found: {OUTPUT_PATH}")
    print("   Did you run Cell 5 successfully?")
else:
    file_size_mb = os.path.getsize(OUTPUT_PATH) / (1024 * 1024)
    print(f"📹 Output: {OUTPUT_PATH}")
    print(f"   Size  : {file_size_mb:.1f} MB")

    # Display inline
    display(HTML(f"""
    <div style="text-align:center; background:#111; padding:20px; border-radius:12px;">
      <p style="color:#aaa; font-size:0.9em; margin-bottom:10px;">
        🎙️ Moshi Voice &nbsp;|&nbsp; 🎬 Ditto Face Animation
      </p>
      <video width="640" height="480" controls autoplay loop
             style="border-radius:8px; box-shadow:0 4px 20px rgba(0,0,0,0.6);">
        <source src="{OUTPUT_PATH}" type="video/mp4">
        Your browser does not support the video tag.
      </video>
    </div>
    """))

---
## 🔄 Cell 7 — Run With New Inputs (Quick Re-run)

After the first run, models stay in GPU memory. Use this cell to quickly process different audio/image inputs **without reloading** anything.

In [ ]:
# ✏️ Change these for your new inputs
NEW_AUDIO  = "4.wav"
NEW_IMAGE  = "obama.jpeg"
NEW_OUTPUT = "4.mp4"

# Reuse loaded models — fast re-run
from unified_pipeline import run_pipeline
import torch

# Note: This will reload models. For true fast re-run, copy the Cell 5
# code above directly and reuse moshi_runner / bridge_runner / ditto_runner.
run_pipeline(
    audio_input     = NEW_AUDIO,
    image_input     = NEW_IMAGE,
    output_path     = NEW_OUTPUT,
    bridge_ckpt     = BRIDGE_CKPT,
    bridge_config   = BRIDGE_CONFIG,
    ditto_data_root = DITTO_DATA_ROOT,
    ditto_cfg_pkl   = DITTO_CFG_PKL,
    moshi_hf_repo   = MOSHI_HF_REPO,
    device          = DEVICE,
    dtype           = DTYPE,
    batch_size      = BATCH_SIZE,
    batch_index     = BATCH_INDEX,
)

---
## 🛠️ Cell 8 — Troubleshooting & Debug

Run individual pipeline steps in isolation to debug issues.

In [ ]:
# ── Test 1: Check Moshi output tokens ────────────────────────────────────────
print("🔬 Moshi token stats:")
if 'acoustic_tokens' in dir():
    print(f"  Shape : {acoustic_tokens.shape}")
    print(f"  Dtype : {acoustic_tokens.dtype}")
    print(f"  Range : [{acoustic_tokens.min().item()}, {acoustic_tokens.max().item()}]")
    print(f"  Sample (first 5 steps, all 8 codebooks):")
    print(acoustic_tokens[:5].numpy())
else:
    print("  ⚠️  Run Cell 5 first to generate tokens.")

print()

# ── Test 2: Check Bridge features ────────────────────────────────────────────
print("🔬 Bridge feature stats:")
if 'audio_features' in dir():
    import numpy as np
    print(f"  Shape : {audio_features.shape}")
    print(f"  Dtype : {audio_features.dtype}")
    print(f"  Mean  : {audio_features.mean():.4f}")
    print(f"  Std   : {audio_features.std():.4f}")
    print(f"  Range : [{audio_features.min():.4f}, {audio_features.max():.4f}]")
    has_nan = np.isnan(audio_features).any()
    print(f"  NaN?  : {has_nan}")
    if has_nan:
        print("  ❌ NaN detected in features — check bridge checkpoint!")
    else:
        print("  ✅ Features look healthy.")
else:
    print("  ⚠️  Run Cell 5 first to generate features.")

In [ ]:
# ── GPU Memory Status ─────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    alloc  = torch.cuda.memory_allocated(0) / 1e9
    reserv = torch.cuda.memory_reserved(0)  / 1e9
    total  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🔬 GPU Memory ({torch.cuda.get_device_name(0)}):")
    print(f"  Allocated : {alloc:.2f} GB")
    print(f"  Reserved  : {reserv:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserv:.2f} GB")
else:
    print("CUDA not available.")

---
<div style="background: #0f3460; padding: 20px; border-radius: 10px; text-align: center;">
  <h3 style="color: #e94560; margin: 0;">⚡ Quick Reference</h3>
  <table style="color: #a4b9d3; margin: 15px auto; border-collapse: collapse;">
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Install deps</td>
        <td style="padding: 5px 20px;">→ Cell 1 (once per session)</td></tr>
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Check GPU</td>
        <td style="padding: 5px 20px;">→ Cell 2</td></tr>
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Set paths</td>
        <td style="padding: 5px 20px;">→ Cell 3 (edit AUDIO_INPUT, IMAGE_INPUT)</td></tr>
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Load models</td>
        <td style="padding: 5px 20px;">→ Cell 4 (once per session)</td></tr>
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Run pipeline</td>
        <td style="padding: 5px 20px;">→ Cell 5 🚀</td></tr>
    <tr><td style="padding: 5px 20px; text-align:right; font-weight:bold;">Preview video</td>
        <td style="padding: 5px 20px;">→ Cell 6</td></tr>
  </table>
</div>